# GENOME: Recombination Validation Demo

**Hypothesis:** DNA-inspired recombination operators on sentence embeddings produce semantically meaningful hybrids that beat naive averaging on retrieval tasks.

This notebook reproduces the end-to-end evaluation: parent pairs â†’ embeddings â†’ recombination operators â†’ retrieval from a corpus â†’ precision@k metrics â†’ visualizations.

**Reference docs:**
- [Design doc](../docs/plans/2026-04-22-genome-recombination-design.md)
- [Implementation plan](../docs/plans/2026-04-22-genome-recombination-implementation.md)

## 1. Setup

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from genome.embeddings import EmbeddingProvider
from genome.dataset import load_parent_pairs
from genome.corpus import build_default_corpus
from genome.operators import OPERATORS
from genome.evaluate import run_evaluation
from genome.viz import plot_operator_bar_chart, plot_parent_hybrid_tsne

provider = EmbeddingProvider()
print(f"Embedding model: {provider.model_name} (dim={provider.dim})")

## 2. Dataset

100 curated parent pairs with expected hybrid concepts.

In [ ]:
pairs = load_parent_pairs()
print(f"Loaded {len(pairs)} parent pairs")
pd.DataFrame([
    {"id": p.id, "parent_a": p.parent_a, "parent_b": p.parent_b, "expected_hybrids": ', '.join(p.expected_hybrids)}
    for p in pairs[:10]
])

## 3. Operators

In [ ]:
print("Available recombination operators:")
for name in OPERATORS.keys():
    print(f"  - {name}")

## 4. Single-pair demo

Apply every operator to one pair and inspect top-5 retrievals.

In [ ]:
corpus = build_default_corpus(provider=provider)
print(f"Corpus size: {len(corpus)}")

pair = pairs[0]  # ML engineer + product manager
print(f"\nParent A: {pair.parent_a}")
print(f"Parent B: {pair.parent_b}")
print(f"Expected hybrids: {pair.expected_hybrids}")

a_vec = provider.encode(pair.parent_a)
b_vec = provider.encode(pair.parent_b)

rows = []
for op_name, op in OPERATORS.items():
    hybrid = op(a_vec, b_vec)
    top5 = corpus.search(hybrid, k=5)
    rows.append({"operator": op_name, "top-5 retrievals": " | ".join(r.text for r in top5)})

pd.DataFrame(rows)

## 5. Full evaluation

Run all operators across all 100 pairs. Takes ~30-60s on CPU.

In [ ]:
results = run_evaluation(provider=provider, corpus=corpus)

df = pd.DataFrame(results).T
df = df.sort_values("precision@5", ascending=False)
df

## 6. Bar chart

In [ ]:
plot_operator_bar_chart(results, metric="precision@5", output="../results/bar_chart.png")
from IPython.display import Image
Image("../results/bar_chart.png")

## 7. t-SNE of parents and hybrids

Using `uniform_crossover` on 10 pairs.

In [ ]:
from genome.operators import uniform_crossover

sample = pairs[:10]
parents_a = np.array([provider.encode(p.parent_a) for p in sample])
parents_b = np.array([provider.encode(p.parent_b) for p in sample])
hybrids = np.array([uniform_crossover(a, b, seed=42) for a, b in zip(parents_a, parents_b)])
labels = [p.id for p in sample]

plot_parent_hybrid_tsne(parents_a, parents_b, hybrids, labels, output="../results/tsne.png")
Image("../results/tsne.png")

## 8. Conclusion

On 100 parent pairs with a 370-item retrieval corpus (hybrids + parents + distractors), recombination operators show a small but real advantage over averaging:

- `frequency_crossover` was the top operator, narrowly beating `simple_average`
- `uniform_crossover` and `uniform_crossover_with_mutation` also edged averaging
- `concat_project` (random-projection baseline) collapses to near zero, confirming that random projection destroys semantic structure
- Single-point and multi-point crossover underperformed, likely because they create sharp discontinuities in the embedding space

Full numbers and discussion in `docs/writeup.md`.